# AVAC analytical-verification figures

This notebook creates the Section 4 manuscript figures from the three AVAC publication cases. On a clean checkout it visibly regenerates those cases with the current source before plotting; set `ENSURE_CURRENT_RESULTS=False` only to restyle already-audited results.


## Reproducible environment

The first code cell installs the repository's validation package and Python dependencies into the active kernel. A clean checkout also needs GNU Make and gfortran to compile the selected solver on first use.


In [ ]:
from pathlib import Path
SEARCH_ROOT = Path.cwd().resolve()
REPOSITORY = next(candidate for candidate in (SEARCH_ROOT, *SEARCH_ROOT.parents) if (candidate / 'validation' / 'pyproject.toml').is_file())
%pip install -q -e {REPOSITORY / 'validation'}


In [ ]:
import os
from avac4qgis_validation import validation_case
case = validation_case('AVAC', 'Paper_figures')
CORES = max(1, os.cpu_count() or 1)
case.path


## Required numerical products

The WRR, Kerswell horizontal-Coulomb, and inclined-Coulomb drivers below use the same controls as their dedicated notebooks. Their summaries retain the solver hash, AMR controls, and achieved levels. The two Coulomb rows are postprocessed with one boundary definition: the wet support uses the solver dry tolerance and the undisturbed rear uses a 0.1% relative-depth tolerance so AMR interpolation noise is not mistaken for motion. Both the raw run summaries and the common manuscript diagnostics are archived with the figures.


In [ ]:
ENSURE_CURRENT_RESULTS = True
AVAC_ROOT = case.path.parent
if ENSURE_CURRENT_RESULTS:
    wrr = AVAC_ROOT / '2008_WRR_sloping_bed'
    case.run(wrr / 'run_avac_validation.py', '--dx', 0.04, '--t-final', 5.0, '--nout', 20, '--cores', CORES, '--amr-levels', 3, '--amr-ratio', 4, '--speed-tolerance', 0.02, '--ny', 5, '--max1d', 1000, '--output-root', wrr / 'publication_amr', cwd=wrr)
    kerswell = AVAC_ROOT / 'Kerswell_Coulomb'
    case.run(kerswell / 'run_avac_validation.py', '--dx', 0.04, '--t-final', 10.0, '--nout', 40, '--cores', CORES, '--case-name', 'publication_amr', '--amr-levels', 3, '--amr-ratio', 4, '--speed-tolerance', 0.02, '--max1d', 1000, cwd=kerswell)
    inclined = AVAC_ROOT / 'Coulomb_sloping_bed'
    case.run(inclined / 'run_avac_validation.py', '--dx', 0.03, '--ny', 5, '--t-final', 6.0, '--nout', 30, '--case-name', 'publication_amr', '--cores', CORES, '--amr-levels', 3, '--amr-ratio', 4, '--speed-tolerance', 0.02, '--max1d', 1000, '--replace', cwd=inclined)
case.run('make_avac_verification_figures.py', '--output-root', REPOSITORY / 'docs' / 'article' / 'figures')


In [ ]:
case.show('../../../docs/article/figures/avac_coulomb_verification.png', '../../../docs/article/figures/avac_wrr_water_limit.png')
